# MINC Conversion

In [ ]:
from pathlib import Path

REPO_DIR = Path.cwd()
while not (REPO_DIR / 'pyproject.toml').exists():
    REPO_DIR = REPO_DIR.parent
print(f'Repository root: {REPO_DIR}')

In [ ]:
import gzip
import json
import shutil
import tempfile
from pathlib import Path

from brainweb_denoising.minc_conversion import convert_minc_to_nifti

In [ ]:
MINC_DIR = (REPO_DIR / 'data/raw/minc')
NIFTI_DIR = (REPO_DIR / 'data/raw/nifti')
NIFTI_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
reports = []
with tempfile.TemporaryDirectory() as tmp:
    for gz_path in sorted(MINC_DIR.glob('*.mnc.gz')):
        uncompressed = Path(tmp) / (gz_path.name[:-len('.mnc.gz')] + '.mnc')
        with gzip.open(gz_path, 'rb') as src, open(uncompressed, 'wb') as dst:
            shutil.copyfileobj(src, dst)
        final = convert_minc_to_nifti(
            uncompressed,
            NIFTI_DIR / (uncompressed.stem + '.nii.gz'),
            report_path=(REPO_DIR / 'results/tables/conversion_reports') / (uncompressed.stem + '.json'),
        )
        reports.append(json.loads(((REPO_DIR / 'results/tables/conversion_reports') / (uncompressed.stem + '.json')).read_text()))
        print(f'converted {gz_path.name} -> {final.name}')

(REPO_DIR / 'results/tables/conversion_report.json').write_text(json.dumps(reports, indent=2))
print(f'{len(reports)} volumes converted, {sum(r["status"] == "success" for r in reports)} successful')